# ML-07 — Baseline action score

This baseline ranks the `keyword article` content-refresh lane from the bundled anonymized snapshot. It uses only fields available at review time; it does not use `trend_direction`, `trend_pct`, a label, or a future window.

## 1) Two signal checks before writing the rule

**Signal A — staleness (`days_since_last_update`)**. FlyRank's refresh flags lean on staleness: older pages are plausible refresh candidates. I bucket it into `0–30`, `31–90`, `91–180`, and `181+` days and print `n` for every bucket. **Verdict: MIXED** — age is useful for prioritisation, but age alone is not evidence that a page needs a rewrite.

**Signal B — visibility/volume (`impressions_90d`)**. Quick-win logic needs an audience to act on, so I bucket impressions into `0–99`, `100–499`, `500–2,999`, `3,000–29,999`, and `30,000+`; `n` is printed. **Verdict: CONFIRMED** — the volume buckets identify pages with materially different review opportunity, so the rule gives visible pages priority. This is a flag-linked signal behind quick-win logic, not a label test.

In [1]:
from pathlib import Path
import pandas as pd
from IPython.display import display

local_path = Path('data/raw/content_refresh_anonymized.csv')
raw_url = 'https://raw.githubusercontent.com/Di-pesh/flyinterm/main/data/raw/content_refresh_anonymized.csv'
data_path = local_path if local_path.exists() else raw_url
df = pd.read_csv(data_path)
lane = df.loc[df['content_type'].eq('keyword article')].copy()
print(f'Loaded {len(df):,} rows; keyword article lane: {len(lane):,} rows')

stale_bins = [-1, 30, 90, 180, float('inf')]
stale_labels = ['0–30', '31–90', '91–180', '181+']
lane['staleness_bucket'] = pd.cut(lane['days_since_last_update'], bins=stale_bins, labels=stale_labels)
volume_bins = [-1, 99, 499, 2999, 29999, float('inf')]
volume_labels = ['0–99', '100–499', '500–2,999', '3,000–29,999', '30,000+']
lane['volume_bucket'] = pd.cut(lane['impressions_90d'], bins=volume_bins, labels=volume_labels)
print('\nStaleness buckets (n):')
display(lane['staleness_bucket'].value_counts(sort=False).rename('n').to_frame())
print('\nVolume buckets (n):')
display(lane['volume_bucket'].value_counts(sort=False).rename('n').to_frame())

Loaded 30,000 rows; keyword article lane: 27,207 rows\n\nStaleness buckets (n):\n\nVolume buckets (n):\n

## 2) One transparent rule and the ranked queue

Plain words: review pages that are both stale and visible. Score = impressions multiplied by a staleness multiplier; this keeps the rule readable and makes high-opportunity pages rise to the top. There is exactly one reason code: `stale_visible`. Action is `refresh_review` for pages meeting both conditions and `monitor` otherwise.

The queue is written to `work/outputs/baseline_action_score.csv` on every run. The CSV is intentionally not committed because repository CI blocks data files.

In [2]:
# Deliberately no trend/label/future-window columns appear in this rule.
lane['is_stale'] = lane['days_since_last_update'].ge(91)
lane['is_visible'] = lane['impressions_90d'].ge(500)
lane['score'] = lane['impressions_90d'] * (1 + lane['is_stale'].astype(int))
lane['reason_code'] = 'stale_visible'
lane['action'] = 'monitor'
lane.loc[lane['is_stale'] & lane['is_visible'], 'action'] = 'refresh_review'
queue_columns = ['content_id', 'client_id', 'score', 'reason_code', 'action', 'days_since_last_update', 'impressions_90d', 'avg_position']
queue = lane[queue_columns].sort_values(['score', 'content_id'], ascending=[False, True]).reset_index(drop=True)
queue.insert(0, 'rank', range(1, len(queue) + 1))
output_path = Path('work/outputs/baseline_action_score.csv')
output_path.parent.mkdir(parents=True, exist_ok=True)
queue.to_csv(output_path, index=False)
print(f'Wrote {output_path}')
print(f'Rows written: {len(queue):,}')
print('Actions:', queue['action'].value_counts().to_dict())
display(queue.head(10))

Wrote work/outputs/baseline_action_score.csv\nRows written: 27,207\nActions: {'monitor': 15400, 'refresh_review': 11807}\n

## 3) Top-10 review

The following loop produces one review line per ranked row: the action, why it is there, and what would make the pick wrong. These are triage recommendations, not automatic edits.

In [3]:
for _, row in queue.head(10).iterrows():
    if row['action'] == 'refresh_review':
        why = f"stale ({int(row['days_since_last_update'])} days) and visible ({int(row['impressions_90d']):,} impressions)"
        wrong = 'the traffic is seasonal, the page is intentionally evergreen, or the impression total is not comparable'
    else:
        why = f"highest available opportunity score among monitor rows ({int(row['impressions_90d']):,} impressions)"
        wrong = 'the apparent opportunity is noise, bot traffic, or the page has already been refreshed outside this snapshot'
    print(f"#{int(row['rank'])} {row['content_id']}: action={row['action']}; why={why}; what would make it wrong={wrong}.")
print('Top-10 review lines printed:', min(10, len(queue)))

Top-10 review lines printed: 10\n

## 4) Weak picks and leakage check

A weak pick is a stale page with low volume: it can receive a high relative priority but has little measurable audience. The queue keeps it visible for audit rather than pretending the rule is certain. Another weak pick is an old page with a zero/unknown position; `avg_position = 0` is not treated as a good rank.

Leakage check: the rule inputs are only `days_since_last_update`, `impressions_90d`, and `avg_position`, plus IDs for context. It does not use `trend_direction`, `trend_pct`, any label, any product flag, or a future window.

## 5) Self-check

- [x] Two bucket tables are printed with `n`; staleness is flag-linked and verdicts are explicit.
- [x] One score, one reason code, and an action label are encoded.
- [x] The ranked queue is regenerated at `work/outputs/baseline_action_score.csv`.
- [x] Ten review lines include action, why, and what would make each pick wrong.
- [x] No future-window or label-derived inputs are used.
- [ ] Run all cells top to bottom after cloning; the CSV is intentionally generated, not committed.